# PondPredict: 08 Recursive Multi-Step Forecasting (1–24 Hours)

**Requested by thesis adviser: 12-hour and 24-hour experimental forecasting, not app integration.**

This notebook uses the **already-trained final** Philippine-only and transfer-learning LSTM checkpoints with the same 6-hour input window and fitted Philippine MinMaxScaler. No retraining, fine-tuning, new input variables, or hyperparameter changes occur.

**Leakage-safe evaluation:** The Philippine observations are chronologically divided into **201 training / 43 validation / 44 held-out test** rows. All recursive inputs and actual targets are drawn only from the 44-row test segment, with **15 common forecast origins** for horizons +1 through +24. Predicted scaled pH/temperature pairs are fed into the rolling window; actual future values are used *only after inference* for scoring.

Exports: 720 prediction records, 96 metric rows (2 models × 2 targets × 24 horizons), 48 model comparisons, 2 error graphs, checkpoint verification and SHA-256 manifest. The outputs support Chapters 3–5. Because origins overlap and this is a short single-pond test set, these are exploratory results, not operational guarantees.

In [1]:
"""PondPredict: reproducible 1–24h recursive evaluation (no model training).

Run from the project root, or place this script there. Saves results/recursive_24h/.
The 15 common origins are restricted to the original 44-row held-out test block.
"""

'PondPredict: reproducible 1–24h recursive evaluation (no model training).\n\nRun from the project root, or place this script there. Saves results/recursive_24h/.\nThe 15 common origins are restricted to the original 44-row held-out test block.\n'

## Setup and source files

In [2]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # Must precede keras import.
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import joblib
import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "PH_data.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "PH_data.csv"
SCALER_FILE = ROOT / "models" / "ph_scaler.pkl"
MODEL_FILES = {
    "Philippine-only": ROOT / "models" / "ph_baseline.keras",
    "Transfer learning": ROOT / "models" / "transfer_final.keras",
}
OUT = ROOT / "results" / "recursive_24h"
SEQUENCE_LENGTH, MAX_HORIZON = 6, 24
SEED = 42
TARGETS = [("pH", "pH"), ("temperature", "temperature_C")]
EXPECTED_1H_MAE = {
    "Philippine-only": (0.0227735459, 0.1870752736),
    "Transfer learning": (0.0229779133, 0.1581814716),
}


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_checkpoint(path, transfer=False):
    try:
        return keras.models.load_model(path, compile=False)
    except (TypeError, ValueError) as exc:
        from keras import Sequential
        from keras.layers import Input, LSTM, Dropout, Dense
        names = (["transfer_lstm", "transfer_dropout", "transfer_output"] if transfer
                 else ["lstm", "dropout", "dense"])
        model = Sequential([
            Input(shape=(SEQUENCE_LENGTH, 2)),
            LSTM(16, name=names[0]),
            Dropout(0.20, name=names[1]),
            Dense(2, name=names[2]),
        ])
        model.load_weights(path)
        warnings.warn("Loaded saved weights into identical architecture because this Keras version "
                      f"could not deserialize the original config: {str(exc)[:160]}")
        return model

## Chronological split and original scaler

In [3]:
def load_test_data():
    raw = pd.read_csv(DATA).rename(columns={
        "Date & Time": "created_date", "pH": "water_pH", "Water Temp (°C)": "water_temp"
    })
    required = ["created_date", "water_pH", "water_temp"]
    assert set(required).issubset(raw.columns), f"Missing columns: {required}"
    raw["created_date"] = pd.to_datetime(raw["created_date"], format="%Y-%m-%d %H:%M")
    raw = raw.sort_values("created_date").reset_index(drop=True)
    assert not raw[required].isna().any().any(), "Missing data"
    assert not raw.created_date.duplicated().any(), "Duplicate timestamps"
    assert raw.created_date.diff().dropna().eq(pd.Timedelta(hours=1)).all(), "Hourly gap"
    assert len(raw) == 288
    split_train, split_test = int(288 * .70), int(288 * .85)
    assert (split_train, split_test) == (201, 244)
    test_df = raw.iloc[split_test:].reset_index(drop=True)
    assert len(test_df) == 44
    scaler = joblib.load(SCALER_FILE)  # Trusted project-provided file only.
    observed = test_df[["water_pH", "water_temp"]].to_numpy(dtype=float)
    scaled = np.asarray(scaler.transform(observed), dtype=np.float32)
    return raw, test_df, scaler, observed, scaled

## Verify final saved checkpoint identity via scores

In [4]:
def verify_38_one_hour(test_scaled, test_original, scaler, models):
    inputs = np.stack([test_scaled[i:i+SEQUENCE_LENGTH]
                       for i in range(len(test_scaled)-SEQUENCE_LENGTH)]).astype(np.float32)
    actuals = test_original[SEQUENCE_LENGTH:]
    assert len(inputs) == len(actuals) == 38
    rows = []
    for label, model in models.items():
        pred = scaler.inverse_transform(np.asarray(model.predict(inputs, verbose=0), dtype=np.float32))
        for idx, (target, _) in enumerate(TARGETS):
            mae = float(mean_absolute_error(actuals[:, idx], pred[:, idx]))
            expected = EXPECTED_1H_MAE[label][idx]
            assert abs(mae-expected) < 0.0005, (label,target,mae,expected)
            rows.append({"model":label,"target":target,"test_targets":38,
                         "MAE":mae,"expected_final_MAE":expected,"matches_final":True})
    return pd.DataFrame(rows)

## Predict 24 hours recursively from 15 held-out origins

In [5]:
def recursive_batch(model, windows, horizons=MAX_HORIZON):
    window = np.asarray(windows, dtype=np.float32).copy()
    all_predictions = []
    for h in range(1, horizons+1):
        scaled_next = np.asarray(model.predict(window, verbose=0), dtype=np.float32)
        assert scaled_next.shape == (len(windows), 2)
        assert np.isfinite(scaled_next).all(), f"Non-finite prediction at +{h}h"
        all_predictions.append(scaled_next)
        window = np.concatenate([window[:, 1:, :], scaled_next[:,None,:]], axis=1)
    return np.stack(all_predictions, axis=1)


def build_forecast_dataframe(test_df, test_scaled, test_original, scaler, models):
    n_origins = len(test_df) - SEQUENCE_LENGTH - MAX_HORIZON + 1
    assert n_origins == 15, f"Expected 15 common origins; got {n_origins}"
    windows = np.stack([test_scaled[i:i+SEQUENCE_LENGTH]
                        for i in range(n_origins)]).astype(np.float32)
    actual_cube = np.stack([test_original[i+SEQUENCE_LENGTH:i+SEQUENCE_LENGTH+MAX_HORIZON]
                            for i in range(n_origins)])
    assert actual_cube.shape == (15,24,2)
    rows = []
    for label, model in models.items():
        scaled_cube = recursive_batch(model, windows)
        predicted_cube = scaler.inverse_transform(scaled_cube.reshape(-1,2)).reshape(15,24,2)
        for origin_idx in range(n_origins):
            origin_time = test_df.created_date.iloc[origin_idx+SEQUENCE_LENGTH-1]
            for h in range(1,MAX_HORIZON+1):
                target_time = test_df.created_date.iloc[origin_idx+SEQUENCE_LENGTH+h-1]
                assert target_time-origin_time == pd.Timedelta(hours=h)
                a, p = actual_cube[origin_idx,h-1],predicted_cube[origin_idx,h-1]
                rows.append({
                    "model":label,"origin_index":origin_idx,
                    "forecast_origin":origin_time,"forecast_timestamp":target_time,
                    "horizon_hours":h,
                    "actual_pH":float(a[0]),"predicted_pH":float(p[0]),
                    "absolute_error_pH":float(abs(a[0]-p[0])),
                    "actual_temperature_C":float(a[1]),"predicted_temperature_C":float(p[1]),
                    "absolute_error_temperature_C":float(abs(a[1]-p[1])),
                })
    result = pd.DataFrame(rows)
    assert len(result) == 2*15*24 == 720
    assert result.notna().all().all()
    assert result.groupby(["model","horizon_hours"]).size().eq(15).all()
    assert all(result.groupby("origin_index").forecast_origin.nunique() == 1)
    for h in range(1,25):
        b = result[(result.model=="Philippine-only")&(result.horizon_hours==h)].sort_values("origin_index")
        t = result[(result.model=="Transfer learning")&(result.horizon_hours==h)].sort_values("origin_index")
        assert np.allclose(b.actual_pH, t.actual_pH)
        assert np.allclose(b.actual_temperature_C, t.actual_temperature_C)
    return result

## Calculate MAE, RMSE, R² and comparisons

In [6]:
def compute_metrics(forecasts):
    rows=[]
    for (model,h), group in forecasts.groupby(["model","horizon_hours"],sort=True):
        for target, suffix in TARGETS:
            truth = group[f"actual_{suffix}"].to_numpy(dtype=float)
            prediction = group[f"predicted_{suffix}"].to_numpy(dtype=float)
            rows.append({
                "model":model,"target":target,"horizon_hours":int(h),
                "forecast_origins":len(group),"MAE":float(mean_absolute_error(truth,prediction)),
                "RMSE":float(np.sqrt(mean_squared_error(truth,prediction))),
                "R2":float(r2_score(truth,prediction)),
            })
    metrics = pd.DataFrame(rows).sort_values(["target","horizon_hours","model"]).reset_index(drop=True)
    assert len(metrics) == 2*2*24 == 96
    assert metrics.forecast_origins.eq(15).all()
    comparisons = []
    for (target,h),g in metrics.groupby(["target","horizon_hours"],sort=True):
        base = g[g.model=="Philippine-only"].iloc[0]
        transfer = g[g.model=="Transfer learning"].iloc[0]
        comparisons.append({
            "target":target,"horizon_hours":int(h),"forecast_origins":15,
            "baseline_MAE":float(base.MAE),"transfer_MAE":float(transfer.MAE),
            "baseline_RMSE":float(base.RMSE),"transfer_RMSE":float(transfer.RMSE),
            "baseline_R2":float(base.R2),"transfer_R2":float(transfer.R2),
            "transfer_MAE_reduction_percent":float(100*(base.MAE-transfer.MAE)/base.MAE) if base.MAE else np.nan,
            "lower_MAE_model":"Transfer learning" if transfer.MAE < base.MAE else (
                "Philippine-only" if base.MAE < transfer.MAE else "Equal"),
        })
    comparison = pd.DataFrame(comparisons)
    assert len(comparison) == 48
    return metrics, comparison

## Generate pH and temperature figures

In [7]:
def generate_graphs(metrics, directory):
    for target, title, ylabel, filename in [
        ("pH","Recursive pH error through 24 hours","MAE (pH units)","recursive_24h_ph_graph.png"),
        ("temperature","Recursive water-temperature error through 24 hours","MAE (°C)","recursive_24h_temperature_graph.png"),
    ]:
        fig,ax=plt.subplots(figsize=(9.2,5.8))
        for label in MODEL_FILES:
            subset=metrics[(metrics.target==target)&(metrics.model==label)].sort_values("horizon_hours")
            ax.plot(subset.horizon_hours, subset.MAE, marker="o",markersize=3,linewidth=1.8,label=label)
        ax.axvline(12, color="gray", linestyle="--",alpha=.6)
        ax.axvline(24, color="gray", linestyle="--",alpha=.6)
        ax.set_xticks([1,3,6,9,12,15,18,21,24])
        ax.set_xlabel("Recursive forecast horizon (hours)")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(alpha=.2)
        ax.legend(loc="best")
        fig.tight_layout()
        fig.savefig(directory/filename,dpi=220,bbox_inches="tight")
        plt.close(fig)

## Save evidence and print findings

In [8]:
def main():
    for file in [DATA, SCALER_FILE, *MODEL_FILES.values()]:
        if not file.is_file():
            raise FileNotFoundError(f"Missing necessary original project file: {file}")
    OUT.mkdir(parents=True,exist_ok=True)
    keras.utils.set_random_seed(SEED)
    print("Backend:", keras.backend.backend(),"Keras:",keras.__version__)
    raw,test,scaler,observed,scaled = load_test_data()
    models={label:load_checkpoint(path,transfer=(label=="Transfer learning"))
            for label,path in MODEL_FILES.items()}
    for label,model in models.items():
        assert tuple(model.input_shape[1:])==(6,2) and tuple(model.output_shape[1:])==(2,),label
    original_1h=verify_38_one_hour(scaled,observed,scaler,models)
    forecasts=build_forecast_dataframe(test,scaled,observed,scaler,models)
    metrics,comparison=compute_metrics(forecasts)
    original_1h.to_csv(OUT/"recursive_24h_checkpoint_verification.csv",index=False)
    forecasts.to_csv(OUT/"recursive_24h_predictions.csv",index=False,float_format="%.12g")
    metrics.to_csv(OUT/"recursive_24h_metrics.csv",index=False,float_format="%.12g")
    comparison.to_csv(OUT/"recursive_24h_comparison.csv",index=False,float_format="%.12g")
    generate_graphs(metrics,OUT)
    manifest={
        "experiment":"24-hour recursive reuse of final 1h models (NO training)",
        "python_environment_backend":"torch","keras_version":keras.__version__,
        "seed_for_inference_reproducibility":SEED,
        "data_rows":len(raw),"train_rows":201,"validation_rows":43,
        "held_out_test_rows":len(test),"test_first_timestamp":str(test.created_date.iloc[0]),
        "test_last_timestamp":str(test.created_date.iloc[-1]),
        "sequence_length":SEQUENCE_LENGTH,"max_horizon":MAX_HORIZON,
        "common_forecast_origins":15,"number_prediction_rows":len(forecasts),
        "number_metric_rows":len(metrics),
        "file_sha256":{str(p.relative_to(ROOT)):sha256(p) for p in [DATA,SCALER_FILE,*MODEL_FILES.values()]},
    }
    (OUT/"recursive_24h_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf8")
    print("\nCHECKPOINT AGREEMENT (all 38 one-hour cases):")
    print(original_1h.to_string(index=False))
    print("\nHORIZON +1/+3/+6/+12/+24 (15 identical origins):")
    print(metrics[metrics.horizon_hours.isin([1,3,6,12,24])].to_string(index=False))
    print("\nDONE: 720 forecast rows, 96 metric rows, 48 comparisons, 2 graphs.")
    print("Results:",OUT)
    return metrics


metrics_result = main()
print("VERIFIED: 24-hour recursive forecast experiment completed.")

Backend: torch Keras: 3.16.0.dev2026092704


c:\Users\Admin\Datas\PondPredict\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



CHECKPOINT AGREEMENT (all 38 one-hour cases):
            model      target  test_targets      MAE  expected_final_MAE  matches_final
  Philippine-only          pH            38 0.022774            0.022774           True
  Philippine-only temperature            38 0.187075            0.187075           True
Transfer learning          pH            38 0.022978            0.022978           True
Transfer learning temperature            38 0.158181            0.158181           True

HORIZON +1/+3/+6/+12/+24 (15 identical origins):
            model      target  horizon_hours  forecast_origins      MAE     RMSE        R2
  Philippine-only          pH              1                15 0.014910 0.018970  0.783741
Transfer learning          pH              1                15 0.020572 0.022019  0.708631
  Philippine-only          pH              3                15 0.017528 0.022515  0.837987
Transfer learning          pH              3                15 0.027517 0.030015  0.712068
  Philip